# Revisión de tablas y outputs del diagnóstico textil Atoyac

Este notebook muestra las tablas generadas por el flujo reproducible. La lectura metodológica es de **priorización preliminar**: identifica actividad textil productiva, confección, maquila, lavado/acabado y proximidad a cauces, sin afirmar contaminación directa.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image, HTML

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists() and (ROOT.parent / 'outputs').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'outputs' / 'tables'
MAPS = ROOT / 'outputs' / 'maps'
FIGURES = ROOT / 'outputs' / 'figures'
INTERACTIVE = ROOT / 'outputs' / 'maps_interactive'

def read_table(name):
    path = TABLES / name
    if not path.exists():
        display(Markdown(f'**No existe:** `{path}`'))
        return pd.DataFrame()
    return pd.read_csv(path)

def show_table(name, n=20):
    df = read_table(name)
    display(Markdown(f'## `{name}`\nFilas: **{len(df):,}** | Columnas: **{len(df.columns)}**'))
    display(df.head(n))
    return df

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
ROOT

## 1. SAIC: lectura municipal y microempresas

SAIC está a nivel municipal/entidad en esta consulta. Sirve para inferir estructura productiva agregada, no para ubicar negocios individuales.

In [ ]:
saic_lectura = show_table('saic_indicadores_lectura_analitica.csv')

In [ ]:
micro = show_table('saic_microempresas_estrato_0_10.csv')

In [ ]:
if not micro.empty:
    cols = ['municipio_nombre','actividad_codigo','estrato','ue','h001a','personal_promedio','proporcion_familiar','maquila_sobre_ingresos']
    display(micro[cols].sort_values(['municipio_nombre','actividad_codigo']))

## 2. DENUE textil productivo: clasificación final

La tabla principal de identificación permite rastrear cada punto del mapa por nombre, CLEE, código SCIAN, categoría y distancia al cauce. El análisis excluye comercio de prendas y conserva producción/confección/maquila/lavado/acabado.

In [ ]:
resumen_denue = show_table('resumen_denue_por_localidad_categoria.csv')

In [ ]:
identificacion = show_table('denue_textil_identificacion.csv', n=25)

In [ ]:
if not identificacion.empty:
    display(Markdown('### Conteo por región y categoría'))
    pivot = identificacion.pivot_table(index='source_folder', columns='categoria_relevancia_ambiental', values='id', aggfunc='count', fill_value=0)
    pivot['total'] = pivot.sum(axis=1)
    display(pivot)

## 3. Auditoría: registros excluidos por comercio o falsos positivos

Esta tabla ayuda a revisar qué se quitó del análisis productivo para no mezclar boutiques, tiendas, comercio de ropa o talleres no textiles.

In [ ]:
excluidos = show_table('denue_excluidos_comercio_prendas.csv', n=30)

## 4. Proximidad a cauces e hidrografía

Los rangos son bandas de proximidad/cautela: no son intervalos estadísticos de confianza ni evidencian contaminación directa.

In [ ]:
rangos = show_table('conteo_negocios_por_rango_distancia.csv', n=50)

In [ ]:
buffers = show_table('conteo_negocios_por_buffer.csv')

## 5. Inventario y control de calidad

In [ ]:
inventario = show_table('inventario_capas.csv', n=15)
errores = show_table('errores_inventario_capas.csv', n=15)

## 6. Outputs cartográficos

In [ ]:
map_files = sorted(MAPS.glob('*.png'))
fig_files = sorted(FIGURES.glob('*.png'))
html_files = sorted(INTERACTIVE.glob('*.html'))

display(Markdown('### Mapas PNG'))
display(pd.DataFrame({'archivo': [p.name for p in map_files], 'ruta': [str(p.relative_to(ROOT)) for p in map_files], 'kb': [round(p.stat().st_size/1024, 1) for p in map_files]}))

display(Markdown('### Figuras'))
display(pd.DataFrame({'archivo': [p.name for p in fig_files], 'ruta': [str(p.relative_to(ROOT)) for p in fig_files], 'kb': [round(p.stat().st_size/1024, 1) for p in fig_files]}))

display(Markdown('### Mapas interactivos HTML'))
display(pd.DataFrame({'archivo': [p.name for p in html_files], 'ruta': [str(p.relative_to(ROOT)) for p in html_files], 'kb': [round(p.stat().st_size/1024, 1) for p in html_files]}))

In [ ]:
for name in ['02_denue_textil_categorias.png','03_buffers_hidrografia_denue.png','07_hidrologia_completa_por_tipo.png','08_hidrologia_completa_por_area_rh.png']:
    path = MAPS / name
    if path.exists():
        display(Markdown(f'### {name}'))
        display(Image(filename=str(path), width=850))

## 7. Enlaces rápidos a mapas interactivos

Abre los HTML desde el explorador de archivos o desde Jupyter para consultar cada punto con popup.

In [ ]:
links = []
for p in sorted(INTERACTIVE.glob('*.html')):
    rel = p.relative_to(ROOT).as_posix()
    links.append(f'- [{p.name}](../{rel})')
display(Markdown('\n'.join(links) if links else 'No hay mapas interactivos.'))